# Day 4 v2 — Model 06: AITeamVN/Vietnamese_Embedding Fine-tune (top-4 layers)

**Architecture:** `AITeamVN/Vietnamese_Embedding` (partial unfreeze: top 4/24 layers) → mean_pooling → price head + aux category head.

**Techniques:** LLRD + EMA (0.999) + Huber Loss + AMP + Cosine Warmup + Multi-task aux head.

**vs Model 05 (frozen encoder, MAE=76.1k):** Fine-tuning top layers should adapt VN embeddings to price-predictive features.

**Target:** MAE < 65k VND

**Dataset:** `SeanSunny/items_tv_v9` (train=269K, val=3926, test=3872)

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [2]:
2

2

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.bert_finetune_model import BERTFinetuneRunner

MODEL_NAME = "AITeamVN/Vietnamese_Embedding"

print(f"Model: {MODEL_NAME}")
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+cu128).


Model: AITeamVN/Vietnamese_Embedding
CUDA: True
GPU: NVIDIA GeForce RTX 3090 Ti
VRAM: 25.3 GB


## 1. Load Data

In [4]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 269,112 | Val: 3,926 | Test: 3,872


## 2. Setup Runner

- Tokenize 269K train + 3926 val (stored as tensors, ~550MB for max_length=256)
- Freeze bottom 20/24 transformer layers, unfreeze top 4
- Configure LLRD param groups: head lr=2e-5, each lower layer × decay=0.9
- Approx trainable params: ~15M encoder + ~1M heads = ~16M total

In [5]:
runner = BERTFinetuneRunner(train, val)

runner.setup(
    model_name=MODEL_NAME,
    keep_top_layers=4,
    batch_size=32,
    max_length=256,
    base_lr=2e-5,
    weight_decay=0.02,
    llrd_decay=0.9,
    dropout=0.2,
)

Loading tokenizer: AITeamVN/Vietnamese_Embedding


config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Categories (8): [np.str_('Bách Hóa'), np.str_('Làm Đẹp - Sức Khỏe'), np.str_('Mẹ và Bé'), np.str_('Nhà Cửa - Đời Sống'), np.str_('Thời Trang'), np.str_('Ô Tô - Xe Máy'), np.str_('Điện Lạnh và Gia Dụng'), np.str_('Điện Tử - Công Nghệ')]
Tokenizing train (269,112) — ~5 min ...
Tokenizing val (3,926) ...
Target: mean=5.4417, std=0.8904 | norm range [-4.10, 1.65]
Loading model: AITeamVN/Vietnamese_Embedding


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Params: 50,657,801/568,027,657 trainable | top 4/24 layers unfrozen | hidden=1024
Device: cuda


## 3. Train

10 epochs, early stopping patience=3. Val MAE evaluated on full 3926 samples per epoch using EMA model.

Expected: ~30-40 min/epoch on RTX 3090 Ti (269K × 256 tokens).

In [6]:
history = runner.train(
    epochs=10,
    patience=3,
    huber_delta=1.0,
    aux_alpha=0.1,
    ema_decay=0.999,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
)

Train: batch=32 | steps/ep=8410 | total=84100 | warmup=8410
LLRD: base=2.0e-05, decay=0.9, emb_lr=1.60e-06 | EMA=0.999 | Huber delta=1.0 | aux_alpha=0.1 | R-Drop=0.0


Epoch  1/10 (1312s) | train_loss=0.2812 | val_loss=0.1553 | val_mae=104.42k | lr=2.00e-05
  ** best val_mae=104.42k


Epoch  2/10 (1315s) | train_loss=0.1290 | val_loss=0.1225 | val_mae=90.15k | lr=1.94e-05
  ** best val_mae=90.15k


Epoch  3/10 (1317s) | train_loss=0.0753 | val_loss=0.1147 | val_mae=85.72k | lr=1.77e-05
  ** best val_mae=85.72k


Epoch  4/10 (1315s) | train_loss=0.0488 | val_loss=0.1111 | val_mae=84.23k | lr=1.50e-05
  ** best val_mae=84.23k


...

Epoch  6/10 (1317s) | train_loss=0.0255 | val_loss=0.1069 | val_mae=82.10k | lr=8.26e-06
  ** best val_mae=82.10k


Epoch  7/10 (1318s) | train_loss=0.0200 | val_loss=0.1060 | val_mae=81.51k | lr=5.00e-06
  ** best val_mae=81.51k


Epoch  8/10 (1319s) | train_loss=0.0166 | val_loss=0.1053 | val_mae=81.19k | lr=2.34e-06
  ** best val_mae=81.19k


Epoch  9/10 (1330s) | train_loss=0.0147 | val_loss=0.1052 | val_mae=81.09k | lr=6.03e-07
  ** best val_mae=81.09k


Epoch 10/10 (1326s) | train_loss=0.0138 | val_loss=0.1051 | val_mae=81.06k | lr=0.00e+00
  ** best val_mae=81.06k
Restored best EMA: val_mae=81.06k


## 4. Training History

In [7]:
plot_training_history(history, title="AITeamVN Fine-tune top-4 layers")

## 5. Save Weights + Val Predictions + Test Predictions

In [8]:
Path("weights").mkdir(exist_ok=True)
runner.save("weights/aitvn_finetune.pth")
print("Saved weights/aitvn_finetune.pth")

Path("val_predictions").mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open("val_predictions/aitvn_finetune_val.json", "w") as f:
    json.dump(val_preds, f)

print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test)
with open("val_predictions/aitvn_finetune_test.json", "w") as f:
    json.dump(test_preds, f)

print(f"Val: {len(val_preds)} | Test: {len(test_preds)}")

Saved weights/aitvn_finetune.pth
Running val predictions (3926 samples)...


Running test predictions (3872 samples)...


Val: 3926 | Test: 3872


## 6. Evaluate on 200 Test Samples

In [9]:
def aitvn_finetune_pricer(item):
    return runner.inference(item)

results = evaluate(aitvn_finetune_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

  0%|          | 0/200 [00:00<?, ?it/s]

280 93 18 76 48 2 58 48 45 6 7 15 284 53 139 89 2 2 104 10 117 63 21 47 10 18 110 24 489 2 75 28 19 13 33 54 46 13 131 182 54 26 121 27 49 329 16 12 38 7 69 68 18 433 61 72 56 16 15 211 2 121 2 13 24 90 3 58 210 4 16 39 53 3 11 65 108 40 47 36 7 519 108 128 48 99 45 3 75 25 10 6 10 418 29 21 14 24 1 24 259 82 9 30 33 27 24 23 21 3 131 109 223 229 39 24 66 252 7 5 176 105 40 55 61 123 16 32 16 0 105 14 112 3 152 114 154 89 276 61 14 1 55 2 233 46 118 24 4 6 208 128 145 221 8 67 9 39 10 10 10 30 42 4 1 104 55 137 24 47 72 9 70 171 14 36 123 0 219 53 112 18 7 7 18 34 29 591 546 1 55 109 16 25 33 31 2 187 16 10 


MAE: 74.2k VND | MSE: 15,635 | R2: 72.8%


## 7. Sanity Check — Load Roundtrip

In [10]:

# Verify checkpoint keys and one sample inference
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred:.1f}k VND")
print(f"Error:   {abs(pred - sample.price):.1f}k VND")

ckpt = torch.load("weights/aitvn_finetune.pth", map_location="cpu", weights_only=False)
expected_keys = {"ema_state_dict", "y_mean", "y_std", "model_name", "keep_top_layers", "cat_classes"}
assert expected_keys == set(ckpt.keys()), f"Unexpected keys: {set(ckpt.keys())}"
print(f"\nCheckpoint keys OK: {sorted(ckpt.keys())}")
print(f"y_mean={ckpt['y_mean']:.4f} | y_std={ckpt['y_std']:.4f} | keep_top_layers={ckpt['keep_top_layers']}")
print(f"Categories ({len(ckpt['cat_classes'])}): {ckpt['cat_classes']}")


Product: Thùng lưu trữ, hộp đựng đồ đa năng bằng nhựa PP cao cấp 30L 
Actual:  479.0k VND
Predict: 199.0k VND
Error:   280.0k VND

Checkpoint keys OK: ['cat_classes', 'ema_state_dict', 'keep_top_layers', 'model_name', 'y_mean', 'y_std']
y_mean=5.4417 | y_std=0.8904 | keep_top_layers=4
Categories (8): ['Bách Hóa', 'Làm Đẹp - Sức Khỏe', 'Mẹ và Bé', 'Nhà Cửa - Đời Sống', 'Thời Trang', 'Ô Tô - Xe Máy', 'Điện Lạnh và Gia Dụng', 'Điện Tử - Công Nghệ']
